# Phase 5: Legal Statute Identification - Encoder-Decoder, Full Fine-tuning

## Task
Legal statute identification: read a case and predict which statutes apply. A case can have several statutes (multi-label). Scored with macro F1.

Model: the encoder of the pre-trained encoder-decoder. The first 2 and last 2 chunks of each case are encoded, their vectors are averaged and one sigmoid per statute gives its score. All parameters are trained.

## Input
- Phase-5 data (needs the HF_TOKEN secret).
- Phase-1 files legal_encdec.pt and tok_encdec.json.

## Flow of the notebook (from input to output)
1. Write the shared helpers: data access, encoder-decoder model, training loop, submission writer.
2. Set the settings, load the model and define how long cases are cut into chunks.
3. Load the case texts, turn the statute lists into 0/1 vectors and cut the cases into chunks.
4. Fine-tune the model and find one threshold on dev (every case gets at least its best statute).
5. Predict the test cases and write the submission zip.

## Output
- Submission zip with the predicted statutes per test case.
- One zip with all outputs.

In [1]:
# Shared code: data, encoder-decoder model, training loop
import os, glob, json, math, time, random, zipfile, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer
DEV, PAD, BOS, EOS, SEP = "cuda" if torch.cuda.is_available() else "cpu", 0, 1, 2, 3
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./out"; os.makedirs(OUT, exist_ok=True)
# It takes rel (file path in the dataset) as input, searches for a file and returns its path.
def find(rel): h = glob.glob(f"./**/{rel}", recursive=True) + glob.glob(f"/kaggle/input/**/{rel}", recursive=True); return h[0] if h else None
# It takes rel (file path in the dataset) as input, gets a dataset file: local copy if present, otherwise download it and returns a local file path.
def getf(rel):
    from huggingface_hub import hf_hub_download
    try: from kaggle_secrets import UserSecretsClient; t = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception: t = os.environ.get("HF_TOKEN", "")
    return find(rel) or hf_hub_download("Exploration-Lab/CS-781-Capstone", rel, repo_type="dataset", token=t)
jload = lambda rel: json.load(open(getf(rel), encoding="utf-8"))
# It takes rel (file path in the dataset) as input, loads a JSON file as a list of records and returns a list of records.
def rec(rel):
    d = jload(rel)
    if isinstance(d, dict) and isinstance(d.get("text"), list): return [d]
    return [{**v, "id": v.get("id", k)} if isinstance(v, dict) else {"id": k, "text": v} for k, v in d.items()] if isinstance(d, dict) else d
class Attn(nn.Module):
    # It takes d (model size) and h (number of heads) as input, creates the attention layer and returns nothing.
    def __init__(s, d, h): super().__init__(); s.h = h; s.q = nn.Linear(d, d); s.kv = nn.Linear(d, 2 * d); s.o = nn.Linear(d, d)
    # It takes x (input tensor), mem (encoder output), mask (attention mask) and causal (use causal mask) as input, runs self-attention or cross-attention and returns output vectors.
    def forward(s, x, mem=None, mask=None, causal=False):
        m = x if mem is None else mem; B, T, D = x.shape; q = s.q(x).view(B, T, s.h, -1).transpose(1, 2); k, v = s.kv(m).view(B, m.size(1), 2, s.h, -1).permute(2, 0, 3, 1, 4)
        return s.o(F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=causal and mask is None).transpose(1, 2).reshape(B, T, D))
class Block(nn.Module):
    # It takes d (model size), h (number of heads), cross (add cross-attention) and p (dropout rate) as input, creates one Transformer block and returns nothing.
    def __init__(s, d, h, cross=False, p=0.1):
        super().__init__(); s.n1, s.sa, s.cross = nn.LayerNorm(d), Attn(d, h), cross
        if cross: s.nx, s.ca = nn.LayerNorm(d), Attn(d, h)
        s.n2, s.ff, s.dp = nn.LayerNorm(d), nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d)), nn.Dropout(p)
    # It takes x (input tensor), mem (encoder output), mm (encoder padding mask), causal (use causal mask) and mask (attention mask) as input, runs one Transformer block and returns output vectors.
    def forward(s, x, mem=None, mm=None, causal=False, mask=None):
        x = x + s.dp(s.sa(s.n1(x), mask=mask, causal=causal))
        if s.cross and mem is not None: x = x + s.dp(s.ca(s.nx(x), mem, mm))
        return x + s.dp(s.ff(s.n2(x)))
class EncDec(nn.Module):
    # It takes V (vocabulary size), d (model size), h (number of heads), ne (encoder layers), nd (decoder layers), L (max length) and p (dropout rate) as input, builds the encoder-decoder model and returns nothing.
    def __init__(s, V=16000, d=640, h=10, ne=6, nd=8, L=1024, p=0.1):
        super().__init__(); s.emb, s.pos = nn.Embedding(V, d), nn.Embedding(L, d); s.enc = nn.ModuleList(Block(d, h, p=p) for _ in range(ne)); s.dec = nn.ModuleList(Block(d, h, True, p) for _ in range(nd)); s.ne_, s.nd_ = nn.LayerNorm(d), nn.LayerNorm(d)
    # It takes x (input tensor) as input, turns token ids into vectors with positions and returns embedding vectors.
    def embed(s, x): return s.emb(x) + s.pos(torch.arange(x.size(1), device=x.device))
    # It takes src (source token ids) as input, runs the encoder over the input text and returns (encoder output, padding mask).
    def encode(s, src):
        mm = (src != PAD)[:, None, None, :]; x = s.embed(src)
        for b in s.enc: x = b(x, mask=mm)
        return s.ne_(x), mm
    # It takes tgt (target token ids), mem (encoder output) and mm (encoder padding mask) as input, runs the decoder and returns next-token scores.
    def decode(s, tgt, mem=None, mm=None):
        x = s.embed(tgt)
        for b in s.dec: x = b(x, mem, mm, causal=True)
        return F.linear(s.nd_(x), s.emb.weight)
    # It takes src (source token ids) and tgt (target token ids) as input, runs encoder then decoder and returns next-token scores (logits).
    def forward(s, src, tgt): return s.decode(tgt, *s.encode(src))
# It loads the pre-trained model, its settings and tokenizer and returns (model, settings, tokenizer).
def base():
    ck = torch.load(find("legal_encdec.pt"), map_location="cpu"); m = EncDec(**ck["cfg"]); m.load_state_dict(ck["sd"]); return m, ck["cfg"], Tokenizer.from_file(find("tok_encdec.json"))
# It takes rows (token lists) and T (length to pad to) as input, pads token lists to the same length and returns a padded tensor of token ids.
def padto(rows, T=None):
    T = T or max(len(r) for r in rows); a = np.full((len(rows), T), PAD, np.int64)
    for i, r in enumerate(rows): a[i, :len(r)] = r[:T]
    return torch.from_numpy(a)
# It takes m (model), ids (token ids) and w (token weights) as input, averages the encoder outputs over the real tokens and returns one vector per row.
def pool(m, ids, w=None):
    h, mm = m.encode(ids); w = (mm[:, 0, 0, :] if w is None else w).to(h.dtype)[..., None]; return (h * w).sum(1) / w.sum(1).clamp(min=1)
# It takes m (model), loss_fn (loss function), sample (batch function), steps (number of steps), lr (learning rate), ev (evaluation function), every (how often), hours (time limit in hours), acc (gradient accumulation steps), hm (hours margin), warm (warm-up steps), log (print function) and fill (progress fill) as input, trains the model with warm-up, cosine learning rate and validation checks and returns training history and the best weights.
def fit(m, loss_fn, sample, steps, lr, ev, every, hours, acc=1, hm=1, warm=100, log=50, fill=False):
    m.to(DEV).train(); ps = [p for p in m.parameters() if p.requires_grad]
    head = [p for n, p in m.named_parameters() if n.startswith("head")]; rest = [p for p in ps if all(p is not h for h in head)]
    opt = torch.optim.AdamW([dict(params=g, mult=k) for g, k in ((rest, 1), (head, hm)) if g], lr=lr, weight_decay=0.01, betas=(0.9, 0.95))
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda"); t0, best, best_state, run, s = time.time(), 1e9, None, 0.0, 0
    while s < steps:
        if s == 25: budget = max(30, int(hours * 3600 / ((time.time() - t0) / 25))); steps = budget if fill else min(steps, budget)
        decay = 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * s / steps))
        for g in opt.param_groups: g["lr"] = lr * g["mult"] * min(1, (s + 1) / warm) * decay
        for _ in range(acc):
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"): loss = loss_fn(m, *sample()) / acc
            scaler.scale(loss).backward(); run += loss.item()
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(ps, 1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); s += 1
        if s % log == 0: print(f"step {s}/{steps} loss {run / log:.4f} | {(time.time() - t0) / 60:.1f} min", flush=True); run = 0.0
        if s % every == 0 or s == steps:
            v = ev(m); m.train(); print(f"   eval {v:.4f} (lower is better)", flush=True)
            if v < best: best, best_state = v, {k: p.detach().cpu().clone() for k, p in m.named_parameters() if p.requires_grad}
    if best_state: m.load_state_dict(best_state, strict=False)
    return m.eval()
# It takes obj (object) and tag (label) as input, writes predictions into a submission zip and returns a path of the submission zip.
def submit(obj, tag):
    p = f"{OUT}/{tag}_prediction.json"; json.dump(obj, open(p, "w"), ensure_ascii=False); z = f"{OUT}/{tag}_submission.zip"
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as f: f.write(p, "prediction.json")
    return z
# It takes name as input, zips all output files into one archive and returns a path of the zip file.
def zip_all(name="ALL_outputs.zip"):
    with zipfile.ZipFile(f"{OUT}/{name}", "w", zipfile.ZIP_DEFLATED) as f:
        for p in sorted(glob.glob(f"{OUT}/*")):
            if os.path.isfile(p) and not p.endswith(name) and not p.endswith(".npy"): f.write(p, os.path.basename(p))
    print(f"{OUT}/{name}", f"{os.path.getsize(f'{OUT}/{name}') / 1e6:.1f} MB")

In [2]:
# Setup: settings, load model, split long documents into chunks
from sklearn.metrics import f1_score
SMOKE = False
BS, ACC, EPOCHS, HOURS, T, LR = 4, 2, 4, 4.5, 512, 1e-4
if SMOKE: BS, ACC, EPOCHS, T = 4, 1, 1, 64
m, cfg, tok = base(); tb = lambda xs: [e.ids for e in tok.encode_batch(xs)]
# It takes texts, size (chunk size), head (number of first chunks) and tail (number of last chunks) as input, cuts a long document into its first and last chunks of tokens and returns (chunks per document, real-chunk flags).
def chunk_ids(texts, size, head=2, tail=2):
    out = []
    for ids in tb(texts):
        ch = [ids[i:i + size] for i in range(0, max(len(ids), 1), size)]; ch = ch if len(ch) <= head + tail else ch[:head] + ch[-tail:]; out.append(ch + [[EOS]] * (head + tail - len(ch)))
    return out, [[len(c) > 1 or c[0] != EOS for c in ch] for ch in out]
# It takes m (model), ch (chunk ids) and ok (real-chunk flags) as input, makes one vector per document from its chunk vectors and returns one vector per document.
def doc_vec(m, ch, ok):
    B, K, T = ch.shape; v = pool(m, ch.view(B * K, T)).view(B, K, -1); o = ok[..., None].to(v.dtype); return (v * o).sum(1) / o.sum(1).clamp(min=1)
# It takes C (chunk token ids), O (real-chunk flags) and idx (indices) as input, pads document chunks into a batch and returns a padded tensors for the batch.
def batch(C, O, idx): return torch.stack([padto(C[i], T) for i in idx]).to(DEV), torch.tensor([O[i] for i in idx]).to(DEV)

In [3]:
# Load case texts and statute labels
tr, dv, te = rec("Phase_5_LSI/Train/train.json"), rec("Phase_5_LSI/Train/dev.json"), rec("Phase_5_LSI/Test/test.json"); print("record keys:", list(tr[0]))
NL = 1 + max(l for r in tr + dv for l in r["labels"])
# It takes rs (records) as input, turns label lists into 0/1 vectors and returns a 0/1 label matrix.
def hot(rs): y = np.zeros((len(rs), NL), np.float32); [y.__setitem__((i, l), 1) for i, r in enumerate(rs) for l in r["labels"]]; return y
(Ct, Ot), (Cd, Od), (Ce, Oe) = [chunk_ids([r["text"] for r in rs], T) for rs in (tr, dv, te)]; yt, yd = hot(tr), hot(dv)
pw = torch.tensor(np.clip((len(yt) - yt.sum(0)) / np.maximum(yt.sum(0), 1), 1, 30), dtype=torch.float32); print(NL, "statute ids | labels per case", yt.sum(1).mean())

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

record keys: ['id', 'text', 'labels']
100 statute ids | labels per case 3.7049825


In [4]:
# Fine-tune the model and find the best threshold on dev
m.head = nn.Linear(cfg["d"], NL); rng = np.random.default_rng(0)
# It picks a random batch of training examples and returns one training batch.
def sample(): idx = rng.integers(0, len(tr), BS); return (*batch(Ct, Ot, idx), torch.tensor(yt[idx]).to(DEV))
# It takes m (model), ch (chunk ids), ok (real-chunk flags) and y (labels) as input, computes the training loss for one batch and returns a loss value.
def lossf(m, ch, ok, y): return F.binary_cross_entropy_with_logits(m.head(doc_vec(m, ch, ok)).float(), y, pos_weight=pw.to(DEV))
# It takes C (chunk token ids) and O (real-chunk flags) as input, gets the model's probability for each statute and returns probabilities per case and statute.
@torch.no_grad()
def probs(C, O):
    m.eval(); out = []
    for j in range(0, len(C), 8):
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"): out += torch.sigmoid(m.head(doc_vec(m, *batch(C, O, range(j, min(j + 8, len(C)))))).float()).tolist()
    return np.array(out)
# It takes p (probabilities) and th (threshold) as input, picks statutes above the threshold (at least one per case) and returns a 0/1 label matrix.
def labels(p, th): y = p > th; y[np.arange(len(p)), p.argmax(1)] = True; return y
mf = lambda y, p: f1_score(y, p, average="macro", zero_division=0)
steps = max(10, EPOCHS * len(tr) // (BS * ACC)); fit(m, lossf, sample, steps, LR, lambda m: -mf(yd, labels(probs(Cd, Od), 0.5)), 250 if not SMOKE else 20, HOURS, ACC, fill=not SMOKE)
pdv = probs(Cd, Od); ths = np.linspace(0.05, 0.9, 35); fs = [mf(yd, labels(pdv, t)) for t in ths]; TH = float(ths[int(np.argmax(fs))])
print(f"dev macro-F1 {max(fs):.4f} at threshold {TH:.2f} | micro-F1 {f1_score(yd, labels(pdv, TH), average='micro'):.4f}")

step 50/44681 loss 1.0735 | 0.3 min
step 100/44681 loss 1.0751 | 0.5 min
step 150/44681 loss 0.9927 | 0.8 min
   eval -0.0689 (lower is better)
step 300/44681 loss 0.9691 | 3.8 min
step 350/44681 loss 0.9360 | 4.1 min
step 400/44681 loss 0.9086 | 4.3 min
step 450/44681 loss 0.8898 | 4.6 min
step 500/44681 loss 0.8903 | 4.9 min
   eval -0.0992 (lower is better)
step 550/44681 loss 0.8400 | 7.4 min
step 600/44681 loss 0.8583 | 7.7 min
step 650/44681 loss 0.8289 | 8.0 min
step 700/44681 loss 0.8062 | 8.3 min
step 750/44681 loss 0.8162 | 8.5 min
   eval -0.1398 (lower is better)
step 800/44681 loss 0.8393 | 11.0 min
step 850/44681 loss 0.7895 | 11.3 min
step 900/44681 loss 0.7889 | 11.6 min
step 950/44681 loss 0.8474 | 11.9 min
step 1000/44681 loss 0.7947 | 12.2 min
   eval -0.1629 (lower is better)
step 1050/44681 loss 0.8037 | 14.7 min
step 1100/44681 loss 0.7763 | 14.9 min
step 1150/44681 loss 0.8285 | 15.2 min
step 1200/44681 loss 0.7421 | 15.5 min
step 1250/44681 loss 0.8073 | 15.8 mi

In [5]:
# Predict on test set and make submission zip
print(submit([{"id": r["id"], "pred": [int(i) for i in np.nonzero(p)[0]]} for r, p in zip(te, labels(probs(Ce, Oe), TH))], f"p5_full_dev{max(fs):.4f}"), len(te), "cases"); zip_all()

/kaggle/working/p5_full_dev0.2796_submission.zip 13019 cases
/kaggle/working/ALL_outputs.zip 0.3 MB
